In [1]:
import torch

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [1]:
from groq import Groq
from dotenv import load_dotenv
import os
import json
import re
from datetime import datetime

In [2]:
load_dotenv()

True

In [3]:
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

print("Setup complete!")


Setup complete!


## Storage Variables

In [4]:
application_data = {
    # Personal Details
    "full_name": None,
    "aadhaar_number": None,
    "pan_number": None,
    "date_of_birth": None,
    "gender": None,
    "category": None,
    "mobile_number": None,
    "email": None,
    "residential_address": None,
    
    # Business Details
    "business_name": None,
    "business_type": None,
    "business_description": None,
    "business_address": None,
    "business_status": None,       # existing or new
    "years_in_operation": None,    # only if existing
    "number_of_employees": None,
    
    # Loan Details
    "loan_category": None,         # Shishu / Kishore / Tarun
    "loan_amount": None,
    "loan_purpose": None,
    "preferred_bank": None,
    
    # Documents
    "documents_provided": []
}

# This holds the conversation history for Claude
conversation_history = []

print("Storage initialized!")
print(json.dumps(application_data, indent=2))

Storage initialized!
{
  "full_name": null,
  "aadhaar_number": null,
  "pan_number": null,
  "date_of_birth": null,
  "gender": null,
  "category": null,
  "mobile_number": null,
  "email": null,
  "residential_address": null,
  "business_name": null,
  "business_type": null,
  "business_description": null,
  "business_address": null,
  "business_status": null,
  "years_in_operation": null,
  "number_of_employees": null,
  "loan_category": null,
  "loan_amount": null,
  "loan_purpose": null,
  "preferred_bank": null,
  "documents_provided": []
}


## Validation Functions

In [5]:
def validate_aadhaar(value):
    value = value.strip().replace(" ", "")
    if not value.isdigit():
        return False, "Aadhaar must contain only numbers"
    if len(value) != 12:
        return False, f"Aadhaar must be exactly 12 digits, you entered {len(value)}"
    return True, "Valid"

def validate_pan(value):
    value = value.strip().upper()
    pattern = r'^[A-Z]{5}[0-9]{4}[A-Z]{1}$'
    if not re.match(pattern, value):
        return False, "PAN must be in format ABCDE1234F (5 letters, 4 numbers, 1 letter)"
    return True, "Valid"

def validate_mobile(value):
    value = value.strip().replace(" ", "")
    if not value.isdigit():
        return False, "Mobile number must contain only digits"
    if len(value) != 10:
        return False, "Mobile number must be exactly 10 digits"
    if value[0] not in ['6', '7', '8', '9']:
        return False, "Mobile number must start with 6, 7, 8, or 9"
    return True, "Valid"

def validate_email(value):
    value = value.strip()
    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    if not re.match(pattern, value):
        return False, "Please enter a valid email address like example@gmail.com"
    return True, "Valid"

def validate_dob(value):
    value = value.strip()
    formats = ["%d/%m/%Y", "%d-%m-%Y", "%Y-%m-%d", "%d %m %Y"]
    for fmt in formats:
        try:
            dob = datetime.strptime(value, fmt)
            age = (datetime.now() - dob).days // 365
            if age < 18:
                return False, f"You must be at least 18 years old to apply. Your age appears to be {age}"
            if age > 100:
                return False, "Please enter a valid date of birth"
            return True, "Valid"
        except ValueError:
            continue
    return False, "Please enter date in DD/MM/YYYY format"

def validate_loan_amount(value, category):
    value = value.strip().replace(",", "").replace("₹", "").replace(" ", "")
    try:
        amount = float(value)
    except ValueError:
        return False, "Please enter a valid number for the loan amount"
    
    if category == "Shishu" and amount > 50000:
        return False, f"Shishu category maximum is ₹50,000. You entered ₹{amount:,.0f}"
    elif category == "Kishore" and (amount <= 50000 or amount > 500000):
        return False, "Kishore category range is ₹50,001 to ₹5,00,000"
    elif category == "Tarun" and (amount <= 500000 or amount > 1000000):
        return False, "Tarun category range is ₹5,00,001 to ₹10,00,000"
    
    return True, "Valid"

def validate_gender(value):
    value = value.strip().lower()
    valid = ["male", "female", "other", "m", "f"]
    if value not in valid:
        return False, "Please enter Male, Female, or Other"
    return True, "Valid"

def validate_category(value):
    value = value.strip().upper()
    valid = ["GENERAL", "SC", "ST", "OBC", "MINORITY"]
    if value not in valid:
        return False, "Please enter one of: General, SC, ST, OBC, Minority"
    return True, "Valid"

def validate_business_type(value):
    value = value.strip().lower()
    valid = ["manufacturing", "trading", "services"]
    if value not in valid:
        return False, "Please enter one of: Manufacturing, Trading, Services"
    return True, "Valid"

def validate_loan_category(value):
    value = value.strip().title()
    valid = ["Shishu", "Kishore", "Tarun"]
    if value not in valid:
        return False, "Please enter one of: Shishu, Kishore, or Tarun"
    return True, "Valid"

print("All validation functions ready!")

All validation functions ready!


## System Prompt

In [ ]:
static_system_prompt = """
You are a helpful government assistant collecting information for a PM Mudra Yojana loan application.

YOUR JOB:
Collect information from the user one question at a time in this exact order:

SECTION 1 - PERSONAL DETAILS:
1. Full name (as per Aadhaar)
2. Aadhaar number (12 digits)
3. PAN number (format: ABCDE1234F)
4. Date of birth (DD/MM/YYYY)
5. Gender (Male/Female/Other)
6. Category (General/SC/ST/OBC/Minority)
7. Mobile number (10 digits)
8. Email address
9. Residential address

SECTION 2 - BUSINESS DETAILS:
10. Business name
11. Type of business (Manufacturing/Trading/Services)
12. Business description (what do they sell/make/do)
13. Business address (ask if same as residential, if yes use that)
14. Is business existing or new?
15. If existing: how many years in operation
16. Number of employees

SECTION 3 - LOAN DETAILS:
17. Loan category:
    - Shishu: up to ₹50,000
    - Kishore: ₹50,001 to ₹5 lakhs  
    - Tarun: ₹5,00,001 to ₹10 lakhs
18. Exact loan amount required
19. Purpose of loan (equipment/working capital/expansion)
20. Preferred bank or lender

SECTION 4 - DOCUMENTS:
21. Inform the user which documents they will need:
    - Aadhaar card (mandatory)
    - PAN card (mandatory)
    - Passport size photo (mandatory)
    - Business proof (only if existing business)
    - Address proof of business (mandatory)
    - Bank statement last 6 months (mandatory)
    - Caste certificate (only if SC/ST/OBC/Minority)
    Then ask them to confirm they have all these ready.

STRICT VALIDATION RULES - THIS IS CRITICAL:
- For Aadhaar number: ONLY accept exactly 12 digits. If wrong, say "That doesn't look like a valid Aadhaar number. It must be exactly 12 digits with no letters or symbols. Please re-enter."
- For PAN number: ONLY accept exactly this format — 5 uppercase letters, 4 digits, 1 uppercase letter (example: ABCDE1234F). If wrong, say "That doesn't look like a valid PAN number. It must follow the format ABCDE1234F. Please re-enter."
- For mobile number: ONLY accept exactly 10 digits starting with 6, 7, 8, or 9. If wrong, reject it.
- NEVER try to fix, guess, correct, or assume what the right value should be for Aadhaar, PAN, and mobile number
- NEVER accept a value that fails validation even if the user insists
- NEVER modify what the user entered to make it fit the format
- Simply ask the user to re-enter the correct value

GENERAL RULES:
- Ask ONE question at a time
- Be friendly and simple, use easy English
- After each answer, acknowledge it warmly before asking next question
- If the user gives multiple answers at once, accept all of them and ask for the next missing field
- If business is NEW, skip the years_in_operation question
- If category is General, skip caste certificate from documents list
- If business address is same as residential, note that and move on
- When all fields are collected, show a complete summary and ask user to confirm or change anything
- ONLY output FORM_COMPLETE followed by JSON when user explicitly says Submit
- NEVER output FORM_COMPLETE until user explicitly confirms submission
"""

print("Static system prompt ready!")

System prompt ready!


## Helper Function to Build the Prompt

This fills in the system prompt with what's been collected so far so LLM always knows the current state.

In [ ]:
def get_fields_needed():
    needed = []
    for key, value in application_data.items():
        if value is None:
            needed.append(key)
    # Remove years_in_operation if business is new
    if application_data["business_status"] == "new":
        if "years_in_operation" in needed:
            needed.remove("years_in_operation")
    return needed

def build_dynamic_prompt():
    collected = {k: v for k, v in application_data.items() if v is not None}
    needed = get_fields_needed()
    
    return f"""
CURRENT APPLICATION STATE:
Fields collected so far: {json.dumps(collected, indent=2)}
Fields still needed: {json.dumps(needed, indent=2)}
"""

print("Dynamic prompt function ready!")

Helper functions ready!


## Validation Router

This takes a field name and value and runs the right validation function

In [8]:
def validate_field(field_name, value):
    validators = {
        "aadhaar_number": validate_aadhaar,
        "pan_number": validate_pan,
        "mobile_number": validate_mobile,
        "email": validate_email,
        "date_of_birth": validate_dob,
        "gender": validate_gender,
        "category": validate_category,
        "business_type": validate_business_type,
        "loan_category": validate_loan_category,
    }
    
    # Loan amount needs special handling (needs category too)
    if field_name == "loan_amount":
        category = application_data.get("loan_category", "Shishu")
        return validate_loan_amount(value, category)
    
    # Fields with no special validation (free text)
    free_text_fields = [
        "full_name", "residential_address", "business_name",
        "business_description", "business_address", "loan_purpose",
        "preferred_bank", "number_of_employees", "years_in_operation",
        "business_status"
    ]
    
    if field_name in free_text_fields:
        if len(value.strip()) < 2:
            return False, "Please provide a valid answer"
        return True, "Valid"
    
    if field_name in validators:
        return validators[field_name](value)
    
    return True, "Valid"

print("Validation router ready!")

Validation router ready!


## Parse LLM's Response and Extract Data

When LLM says FORM_COMPLETE, this function extracts the JSON data from it.

In [9]:
def extract_form_data(response_text):
    if "FORM_COMPLETE" not in response_text:
        return None
    
    try:
        # Find the JSON block after FORM_COMPLETE
        json_start = response_text.find("{", response_text.find("FORM_COMPLETE"))
        json_end = response_text.rfind("}") + 1
        json_str = response_text[json_start:json_end]
        return json.loads(json_str)
    except Exception as e:
        print(f"Could not parse form data: {e}")
        return None

def update_application_data(extracted_data):
    if not extracted_data:
        return
    for key in application_data:
        if key in extracted_data and extracted_data[key]:
            application_data[key] = extracted_data[key]

print("Parser functions ready!")

Parser functions ready!


## Main Chat Function

This is the core function that sends messages to LLM and gets responses

In [ ]:
def chat(user_message):
    # Add user message to history
    conversation_history.append({
        "role": "user",
        "content": user_message
    })
    
    # Call Groq API with static + dynamic prompt and last 3 messages only
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": static_system_prompt + build_dynamic_prompt()}
        ] + conversation_history[-3:],
        max_tokens=1000
    )
    
    assistant_message = response.choices[0].message.content
    
    # Add response to history
    conversation_history.append({
        "role": "assistant",
        "content": assistant_message
    })
    
    # Check if form is complete
    if "FORM_COMPLETE" in assistant_message:
        extracted = extract_form_data(assistant_message)
        update_application_data(extracted)
        clean_response = assistant_message.split("FORM_COMPLETE")[0].strip()
        return clean_response, True
    
    return assistant_message, False

print("Chat function ready!")


## Run the Chatbot

In [11]:
print("=" * 50)
print("PM MUDRA YOJANA APPLICATION BOT")
print("=" * 50)
print("Type 'quit' to exit")
print("Type 'status' to see collected data so far")
print("=" * 50)

# Start the conversation
opening_response, _ = chat("Hello, I want to apply for PM Mudra Yojana loan. Please collect all my information first, show me a summary at the end, and only submit after I confirm.")
print(f"\nBot: {opening_response}\n")

# Main loop
while True:
    user_input = input("You: ").strip()
    
    if not user_input:
        continue
        
    if user_input.lower() == 'quit':
        print("Exiting application. Goodbye!")
        break
    
    if user_input.lower() == 'status':
        print("\n--- COLLECTED DATA SO FAR ---")
        for key, value in application_data.items():
            if value is not None:
                print(f"  {key}: {value}")
        print("-----------------------------\n")
        continue
    
    response, is_complete = chat(user_input)
    print(f"\nBot: {response}\n")
    
    if is_complete:
        print("\n" + "=" * 50)
        print("APPLICATION DATA COLLECTED SUCCESSFULLY")
        print("=" * 50)
        print(json.dumps(application_data, indent=2))
        print("=" * 50)
        print("Ready to be sent to database!")
        break

PM MUDRA YOJANA APPLICATION BOT
Type 'quit' to exit
Type 'status' to see collected data so far

Bot: Hello! I'm happy to help you with your PM Mudra Yojana loan application. I'll guide you through each step, and we'll get all your information collected.

To start, could you please tell me your full name as it appears on your Aadhaar card?



You:  quit


Exiting application. Goodbye!
